# Set Up Cell

In [23]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

# =====================================================
# SET SEED
# =====================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# ROBUST DATASET
# =====================================================
class ImageFolderDataset(Dataset):
    def __init__(self, folder, transform):
        self.paths = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return self.transform(img)
        except:
            # fallback: return a random valid image instead
            new_idx = random.randint(0, len(self.paths) - 1)
            img = Image.open(self.paths[new_idx]).convert("RGB")
            return self.transform(img)

# =====================================================
# MODEL
# =====================================================
def load_model(name):
    if name == "resnet18":
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features

    elif name == "mobilenetv2":
        model = models.mobilenet_v2(weights=None)
        in_features = model.classifier[1].in_features

    elif name == "efficientnetb0":
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features

    else:
        raise ValueError("Unsupported model")

    return model.to(device)


# =====================================================
# FEATURE EXTRACTOR
# =====================================================
def get_feature_extractor(model, name):
    if name == "resnet18":
        return nn.Sequential(*list(model.children())[:-1])
    elif name == "mobilenetv2":
        return model.features
    elif name == "efficientnetb0":
        return model.features

# =====================================================
# EMBEDDINGS
# =====================================================
def extract_embeddings(model, loader, name):
    model.eval()
    extractor = get_feature_extractor(model, name)

    embeddings = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            feats = extractor(batch)
            feats = torch.flatten(feats, 1)
            embeddings.append(feats.cpu().numpy())

    return np.concatenate(embeddings, axis=0)


# =====================================================
# EVALUATION FUNCTION
# =====================================================
def evaluate(model, loader_A, loader_B):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for loader, label in [(loader_A, 0), (loader_B, 1)]:
            for batch in loader:
                batch = batch.to(device)
                outputs = model(batch)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                y_pred.extend(preds)
                y_true.extend([label] * len(preds))

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    return acc, f1

def train_and_evaluate_models(CONFIG):

    set_seed(42)

    os.makedirs(os.path.dirname(CONFIG["output"]["checkpoint_path"]), exist_ok=True)
    
    transform = transforms.Compose([
        transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
        transforms.ToTensor()
    ])

    train_A = ImageFolderDataset(CONFIG["data"]["train_A"], transform)
    train_B = ImageFolderDataset(CONFIG["data"]["train_B"], transform)
    
    val_A = ImageFolderDataset(CONFIG["data"]["val_A"], transform)
    val_B = ImageFolderDataset(CONFIG["data"]["val_B"], transform)
    
    test_A = ImageFolderDataset(CONFIG["data"]["test_A"], transform)
    test_B = ImageFolderDataset(CONFIG["data"]["test_B"], transform)
    
    train_loader_A = DataLoader(train_A, batch_size=CONFIG["batch_size"], shuffle=True)
    train_loader_B = DataLoader(train_B, batch_size=CONFIG["batch_size"], shuffle=True)
    
    val_loader_A = DataLoader(val_A, batch_size=CONFIG["batch_size"], shuffle=False)
    val_loader_B = DataLoader(val_B, batch_size=CONFIG["batch_size"], shuffle=False)
    
    test_loader_A = DataLoader(test_A, batch_size=CONFIG["batch_size"], shuffle=False)
    test_loader_B = DataLoader(test_B, batch_size=CONFIG["batch_size"], shuffle=False)
    
    # =====================================================
    # TRAINING
    # =====================================================
    model = load_model(CONFIG["model_name"])
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    
    for epoch in range(CONFIG["epochs"]):
        print(f"\n🚀 Epoch {epoch+1}/{CONFIG['epochs']}")
    
        model.train()
    
        for batch_A, batch_B in zip(train_loader_A, train_loader_B):
            batch_A = batch_A.to(device)
            batch_B = batch_B.to(device)
    
            inputs = torch.cat([batch_A, batch_B], dim=0)
            labels = torch.cat([
                torch.zeros(len(batch_A)),
                torch.ones(len(batch_B))
            ]).long().to(device)
    
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
        # =====================================================
        # VALIDATION CHECKPOINTING
        # =====================================================
        val_acc, val_f1 = evaluate(model, val_loader_A, val_loader_B)
    
        print(f"Val Accuracy: {val_acc:.4f}, Val F1: {val_f1:.4f}")
    
        if val_acc > best_val_acc:
            best_val_acc = val_acc
    
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc
            }, CONFIG["output"]["checkpoint_path"])
    
            print("✅ Saved new best model!")
    
        model_folder, model_filename = os.path.split(CONFIG["output"]["checkpoint_path"])
        model_filename, ext = os.path.splitext(model_folder)
        epoch_wise_model_filename = model_filename + f'_epoch_{epoch}' + '.' + ext
        epoch_wise_model_filepath = os.path.join(model_folder, epoch_wise_model_filename)
    
        torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc
        }, epoch_wise_model_filepath)
    
    # =====================================================
    # FINAL EVALUATION
    # =====================================================
    train_acc, train_f1 = evaluate(model, train_loader_A, train_loader_B)
    test_acc, test_f1 = evaluate(model, test_loader_A, test_loader_B)
    
    print("\n📊 FINAL RESULTS")
    
    print("\n--- TRAIN ---")
    print(f"Accuracy: {train_acc:.4f}")
    print(f"F1 Score: {train_f1:.4f}")
    
    print("\n--- TEST ---")
    print(f"Accuracy: {test_acc:.4f}")
    print(f"F1 Score: {test_f1:.4f}")


The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# Apples And Oranges

## Resnet-18 Model Training

In [24]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/apples_and_oranges'

CLASS_A = 'apples'
CLASS_B = 'oranges'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_resnet18.pth",
        "plot_path": "/kaggle/working/skin_lesion_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.8500, Val F1: 0.8696
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 5/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 13/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 14/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 15/15
Val Accuracy: 0.9500, Val F1: 0.9524

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9583
F1 Score: 0.9596

--- TEST ---
Accuracy: 0.9372
F1 Score: 0.9366


## MobileNet-v2 Model Training

In [25]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/apples_and_oranges'

CLASS_A = 'apples'
CLASS_B = 'oranges'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_mobilenetv2.pth",
        "plot_path": "/kaggle/working/skin_lesion_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.6000, Val F1: 0.3333
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 0.9000, Val F1: 0.9091

🚀 Epoch 8/15
Val Accuracy: 0.9000, Val F1: 0.9000

🚀 Epoch 9/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 15/15
Val Accuracy: 0.9500, Val F1: 0.9524

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9774
F1 Score: 0.9778

--- TEST ---
Accuracy: 0.9170
F1 Score: 0.9165


## EfficientNet-B0 Model Training

In [26]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/apples_and_oranges'

CLASS_A = 'apples'
CLASS_B = 'oranges'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/apples_and_oranges_efficientnetb0.pth",
        "plot_path": "/kaggle/working/apples_and_oranges_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.8000, Val F1: 0.8333
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 0.8500, Val F1: 0.8235

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9869
F1 Score: 0.9871

--- TEST ---
Accuracy: 0.9291
F1 Score: 0.9275


# Horses And Zebra

## Resnet-18 Model training

In [30]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/horses_and_zebra'

CLASS_A = 'horses'
CLASS_B = 'zebra'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/horses_and_zebra_resnet18.pth",
        "plot_path": "/kaggle/working/horses_and_zebra_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.4667, Val F1: 0.1111
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5333, Val F1: 0.1250
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.5333, Val F1: 0.1250

🚀 Epoch 4/15
Val Accuracy: 0.5667, Val F1: 0.2353
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 0.5667, Val F1: 0.6286

🚀 Epoch 6/15
Val Accuracy: 0.5667, Val F1: 0.3158

🚀 Epoch 7/15
Val Accuracy: 0.5333, Val F1: 0.1250

🚀 Epoch 8/15
Val Accuracy: 0.6667, Val F1: 0.5000
✅ Saved new best model!

🚀 Epoch 9/15
Val Accuracy: 0.5667, Val F1: 0.2353

🚀 Epoch 10/15
Val Accuracy: 0.5333, Val F1: 0.1250

🚀 Epoch 11/15
Val Accuracy: 0.6333, Val F1: 0.5217

🚀 Epoch 12/15
Val Accuracy: 0.5667, Val F1: 0.2353

🚀 Epoch 13/15
Val Accuracy: 0.5667, Val F1: 0.3158

🚀 Epoch 14/15
Val Accuracy: 0.7000, Val F1: 0.7273
✅ Saved new best model!

🚀 Epoch 15/15
Val Accuracy: 0.7333, Val F1: 0.6923
✅ Saved new best model!

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.8071
F1 Score: 0.8278

--- TEST ---
Accur

## Mobiletnet-v2 Model Training

In [31]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/horses_and_zebra'

CLASS_A = 'horses'
CLASS_B = 'zebra'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/horses_and_zebra_mobilenetv2.pth",
        "plot_path": "/kaggle/working/horses_and_zebra_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 3/15
Val Accuracy: 0.6000, Val F1: 0.5714
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.5667, Val F1: 0.2353

🚀 Epoch 5/15
Val Accuracy: 0.7333, Val F1: 0.6923
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 0.5333, Val F1: 0.5000

🚀 Epoch 7/15
Val Accuracy: 0.6000, Val F1: 0.6000

🚀 Epoch 8/15
Val Accuracy: 0.6000, Val F1: 0.3333

🚀 Epoch 9/15
Val Accuracy: 0.6333, Val F1: 0.5926

🚀 Epoch 10/15
Val Accuracy: 0.6333, Val F1: 0.4211

🚀 Epoch 11/15
Val Accuracy: 0.6333, Val F1: 0.6857

🚀 Epoch 12/15
Val Accuracy: 0.6667, Val F1: 0.6154

🚀 Epoch 13/15
Val Accuracy: 0.5667, Val F1: 0.3810

🚀 Epoch 14/15
Val Accuracy: 0.7000, Val F1: 0.5714

🚀 Epoch 15/15
Val Accuracy: 0.5667, Val F1: 0.2353

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9893
F1 Score: 0.9892

--- TEST ---
Accuracy: 0.9042
F1 Score: 0.9098


## EfficientNet-B0 Model Training

In [32]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/horses_and_zebra'

CLASS_A = 'horses'
CLASS_B = 'zebra'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/horses_and_zebra_efficientnetb0.pth",
        "plot_path": "/kaggle/working/horses_and_zebra_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5333, Val F1: 0.1250
✅ Saved new best model!

🚀 Epoch 3/15
Val Accuracy: 0.5667, Val F1: 0.2353
✅ Saved new best model!

🚀 Epoch 4/15
Val Accuracy: 0.6000, Val F1: 0.3333
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 0.6333, Val F1: 0.4211
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 0.6000, Val F1: 0.3333

🚀 Epoch 7/15
Val Accuracy: 0.6000, Val F1: 0.3333

🚀 Epoch 8/15
Val Accuracy: 0.6333, Val F1: 0.5217

🚀 Epoch 9/15
Val Accuracy: 0.6333, Val F1: 0.5217

🚀 Epoch 10/15
Val Accuracy: 0.5333, Val F1: 0.1250

🚀 Epoch 11/15
Val Accuracy: 0.6000, Val F1: 0.3333

🚀 Epoch 12/15
Val Accuracy: 0.6000, Val F1: 0.4000

🚀 Epoch 13/15
Val Accuracy: 0.6333, Val F1: 0.4762

🚀 Epoch 14/15
Val Accuracy: 0.6000, Val F1: 0.4545

🚀 Epoch 15/15
Val Accuracy: 0.5333, Val F1: 0.1250

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9843
F1 Score: 0.9841

--- TEST ---
Accuracy: 0.9250
F1 Score: 0.

# Skin Lesion

In [33]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/skin_lesion'

CLASS_A = 'healthy'
CLASS_B = 'lesion'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_resnet18.pth",
        "plot_path": "/kaggle/working/skin_lesion_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 4/15
Val Accuracy: 0.5500, Val F1: 0.3077
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 0.6500, Val F1: 0.6316
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 0.7000, Val F1: 0.7692
✅ Saved new best model!

🚀 Epoch 7/15
Val Accuracy: 0.5500, Val F1: 0.5263

🚀 Epoch 8/15
Val Accuracy: 0.8000, Val F1: 0.7500
✅ Saved new best model!

🚀 Epoch 9/15
Val Accuracy: 0.9500, Val F1: 0.9474
✅ Saved new best model!

🚀 Epoch 10/15
Val Accuracy: 0.4000, Val F1: 0.4000

🚀 Epoch 11/15
Val Accuracy: 0.9000, Val F1: 0.8889

🚀 Epoch 12/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 13/15
Val Accuracy: 0.6000, Val F1: 0.3333

🚀 Epoch 14/15
Val Accuracy: 0.7500, Val F1: 0.6667

🚀 Epoch 15/15
Val Accuracy: 0.9000, Val F1: 0.8889

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9787
F1 Score: 0.9787

--- TEST ---
Accur

In [34]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/skin_lesion'

CLASS_A = 'healthy'
CLASS_B = 'lesion'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_mobilenetv2.pth",
        "plot_path": "/kaggle/working/skin_lesion_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 4/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 5/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 6/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 7/15
Val Accuracy: 0.5500, Val F1: 0.1818
✅ Saved new best model!

🚀 Epoch 8/15
Val Accuracy: 0.8500, Val F1: 0.8235
✅ Saved new best model!

🚀 Epoch 9/15
Val Accuracy: 0.7500, Val F1: 0.8000

🚀 Epoch 10/15
Val Accuracy: 0.7500, Val F1: 0.8000

🚀 Epoch 11/15
Val Accuracy: 0.9500, Val F1: 0.9474
✅ Saved new best model!

🚀 Epoch 12/15
Val Accuracy: 0.7000, Val F1: 0.6250

🚀 Epoch 13/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 14/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 15/15
Val Accuracy: 0.9000, Val F1: 0.8889

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9702
F1 Score: 0.9698

--- TEST ---
Accuracy: 0.9070
F1 Score: 0.9024


In [35]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/skin_lesion'

CLASS_A = 'healthy'
CLASS_B = 'lesion'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'final_synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/skin_lesion_efficientnetb0.pth",
        "plot_path": "/kaggle/working/skin_lesion_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 4/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 5/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 6/15
Val Accuracy: 0.5000, Val F1: 0.0000

🚀 Epoch 7/15
Val Accuracy: 0.6500, Val F1: 0.4615
✅ Saved new best model!

🚀 Epoch 8/15
Val Accuracy: 0.8000, Val F1: 0.8333
✅ Saved new best model!

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 10/15
Val Accuracy: 0.8500, Val F1: 0.8235

🚀 Epoch 11/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 12/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 13/15
Val Accuracy: 0.8500, Val F1: 0.8235

🚀 Epoch 14/15
Val Accuracy: 0.7000, Val F1: 0.7692

🚀 Epoch 15/15
Val Accuracy: 0.9500, Val F1: 0.9474

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9460
F1 Score: 0.9448

--- TEST ---
Accuracy: 0.8953
F1 Score: 0.8889


# Watermark Dataset

In [36]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/watermark_dataset'

CLASS_A = 'original'
CLASS_B = 'watermarked'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/watermark_resnet18.pth",
        "plot_path": "/kaggle/working/watermark_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 3/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 4/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 6/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 11/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 0.9500, Val F1: 0.9524

🚀 Epoch 14/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9969
F1 Score: 0.9969

--- TEST ---
Accuracy: 1.0000
F1 Score: 1.0000


In [37]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/watermark_dataset'

CLASS_A = 'original'
CLASS_B = 'watermarked'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/watermark_mobilenetv2.pth",
        "plot_path": "/kaggle/working/watermark_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 4/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 5/15
Val Accuracy: 0.9500, Val F1: 0.9524
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 7/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 8/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 9/15
Val Accuracy: 0.9500, Val F1: 0.9474

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.9778
F1 Score: 0.9804


In [38]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/watermark_dataset'

CLASS_A = 'original'
CLASS_B = 'watermarked'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/watermark_efficientnetb0.pth",
        "plot_path": "/kaggle/working/watermark_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 4/15
Val Accuracy: 0.6000, Val F1: 0.7143
✅ Saved new best model!

🚀 Epoch 5/15
Val Accuracy: 1.0000, Val F1: 1.0000
✅ Saved new best model!

🚀 Epoch 6/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 7/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 8/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 9/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 10/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 11/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 12/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 13/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 14/15
Val Accuracy: 1.0000, Val F1: 1.0000

🚀 Epoch 15/15
Val Accuracy: 1.0000, Val F1: 1.0000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 0.9979
F1 Score: 0.9979

--- TEST ---
Accuracy: 1.0000
F1 Score: 1.0000


# Pneumonia CXR

In [39]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/pneumonia_cxr'

CLASS_A = 'NORMAL'
CLASS_B = 'PNEUMONIA'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "resnet18",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/pneumonia_cxr_resnet18.pth",
        "plot_path": "/kaggle/working/pneumonia_cxr_resnet18.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 4/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 5/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 6/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 7/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 8/15
Val Accuracy: 0.5000, Val F1: 0.6000

🚀 Epoch 9/15
Val Accuracy: 0.7500, Val F1: 0.7500
✅ Saved new best model!

🚀 Epoch 10/15
Val Accuracy: 0.7500, Val F1: 0.7143

🚀 Epoch 11/15
Val Accuracy: 0.7500, Val F1: 0.7143

🚀 Epoch 12/15
Val Accuracy: 0.7500, Val F1: 0.7143

🚀 Epoch 13/15
Val Accuracy: 0.7500, Val F1: 0.7143

🚀 Epoch 14/15
Val Accuracy: 0.7500, Val F1: 0.7143

🚀 Epoch 15/15
Val Accuracy: 0.7500, Val F1: 0.7143

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.6138
F1 Score: 0.5704


In [41]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/pneumonia_cxr'

CLASS_A = 'NORMAL'
CLASS_B = 'PNEUMONIA'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "mobilenetv2",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/pneumonia_cxr_mobilenetv2.pth",
        "plot_path": "/kaggle/working/pneumonia_cxr_mobilenetv2.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 4/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 5/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 6/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 7/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 8/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 9/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 10/15
Val Accuracy: 0.6250, Val F1: 0.7273
✅ Saved new best model!

🚀 Epoch 11/15
Val Accuracy: 0.6250, Val F1: 0.7000

🚀 Epoch 12/15
Val Accuracy: 0.5625, Val F1: 0.6316

🚀 Epoch 13/15
Val Accuracy: 0.5625, Val F1: 0.6316

🚀 Epoch 14/15
Val Accuracy: 0.5625, Val F1: 0.6316

🚀 Epoch 15/15
Val Accuracy: 0.6250, Val F1: 0.6667

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.7676
F1 Score: 0.7858


In [42]:
BASE_DIR  = r'/kaggle/input/datasets/medicalradhika/synthetic-data/datasets/pneumonia_cxr'

CLASS_A = 'NORMAL'
CLASS_B = 'PNEUMONIA'

# =====================================================
# CONFIG
# =====================================================
CONFIG = {
    "model_name": "efficientnetb0",
    "epochs": 15,
    "lr": 1e-3,
    "batch_size": 32,
    "image_size": 224,

    "data": {
        "train_A": os.path.join(BASE_DIR, 'synthetic_data', CLASS_A),
        "train_B": os.path.join(BASE_DIR, 'synthetic_data', CLASS_B),
        "val_A": os.path.join(BASE_DIR, 'val_data', CLASS_A),
        "val_B": os.path.join(BASE_DIR, 'val_data', CLASS_B),
        "test_A": os.path.join(BASE_DIR, 'test_data', CLASS_A),
        "test_B": os.path.join(BASE_DIR, 'test_data', CLASS_B),
    },

    "output": {
        "checkpoint_path": "/kaggle/working/pneumonia_cxr_efficientnetb0.pth",
        "plot_path": "/kaggle/working/pneumonia_cxr_efficientnetb0.png"
    }
}

train_and_evaluate_models(CONFIG)


🚀 Epoch 1/15
Val Accuracy: 0.5000, Val F1: 0.6667
✅ Saved new best model!

🚀 Epoch 2/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 3/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 4/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 5/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 6/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 7/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 8/15
Val Accuracy: 0.5000, Val F1: 0.6667

🚀 Epoch 9/15
Val Accuracy: 0.6875, Val F1: 0.7059
✅ Saved new best model!

🚀 Epoch 10/15
Val Accuracy: 0.6875, Val F1: 0.6667

🚀 Epoch 11/15
Val Accuracy: 0.6250, Val F1: 0.5714

🚀 Epoch 12/15
Val Accuracy: 0.6875, Val F1: 0.6667

🚀 Epoch 13/15
Val Accuracy: 0.6250, Val F1: 0.6250

🚀 Epoch 14/15
Val Accuracy: 0.5625, Val F1: 0.4615

🚀 Epoch 15/15
Val Accuracy: 0.6250, Val F1: 0.5000

📊 FINAL RESULTS

--- TRAIN ---
Accuracy: 1.0000
F1 Score: 1.0000

--- TEST ---
Accuracy: 0.6490
F1 Score: 0.6231
